# 🔊 Atomic Step 5: Audio Splitting & Speaker Isolation

This notebook loads the cleaned WAV audio and the refined speaker timeline JSON, slices the audio array into individual speaker segments, and exports them as isolated speaker audio files (concatenating all turns of the same speaker) and/or individual sequence-based turn files.

In [ ]:
# Install dependencies
!pip install -q librosa soundfile scipy
print("[SUCCESS] Dependencies installed!")

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
try:
    drive.mount('/content/drive')
    print("[SUCCESS] Google Drive mounted.")
except Exception:
    print("[INFO] Already mounted or skipped.")

## ⚙️ Parameters

In [ ]:
# @markdown ### 📁 Inputs & Outputs
cleaned_audio_path = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Atomic Notebooks/Cleaned_Audio/MarauliKhurad1_cleaned.wav" # @param {type:"string"}
refined_timeline_json_path = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Atomic Notebooks/Diarization_Outputs/MarauliKhurad1/MarauliKhurad1_timeline.json" # @param {type:"string"}
isolated_output_folder = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Atomic Notebooks/Isolated_Speaker_Audio/" # @param {type:"string"}

# @markdown ### 🔊 Output Splitting Modes
isolate_speaker_wise = True # @param {type:"boolean"}
export_individual_turns = True # @param {type:"boolean"}

if not os.path.exists(cleaned_audio_path):
    print(f"[ERROR] Cleaned audio not found at: '{cleaned_audio_path}'")
elif not os.path.exists(refined_timeline_json_path):
    print(f"[ERROR] Refined timeline JSON not found at: '{refined_timeline_json_path}'")
else:
    audio_filename = os.path.basename(cleaned_audio_path)
    audio_name_only = audio_filename.replace("_cleaned.wav", "").replace(".wav", "")
    os.makedirs(isolated_output_folder, exist_ok=True)
    print(f"[SUCCESS] Validated paths. Isolated audio tracks will be saved in: {isolated_output_folder}")

## 🔊 Split Audio

In [ ]:
import soundfile as sf
import numpy as np
import librosa
import json

if os.path.exists(cleaned_audio_path) and os.path.exists(refined_timeline_json_path):
    # 1. Load refined timeline
    with open(refined_timeline_json_path, "r", encoding="utf-8") as f:
        speaker_segments = json.load(f)
        
    # Create specific folder for output files
    specific_isolated_folder = os.path.join(isolated_output_folder, audio_name_only)
    os.makedirs(specific_isolated_folder, exist_ok=True)
    
    # Load cleaned audio file
    print(f"Loading cleaned audio file for splitting: '{cleaned_audio_path}'")
    y, sr = librosa.load(cleaned_audio_path, sr=16000, mono=True)
    
    # Dictionary to hold samples for concatenating speaker-wise
    speaker_audio_data = {}
    
    # Time formatting helper for filenames: convert seconds to [MM-SS.d]
    def format_time_filename(seconds):
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        millis = int((seconds - int(seconds)) * 10)
        if hours > 0:
            return f"{hours:02d}-{minutes:02d}-{secs:02d}.{millis:01d}"
        else:
            return f"{minutes:02d}-{secs:02d}.{millis:01d}"
            
    # 1. Process and extract segments
    for idx, entry in enumerate(speaker_segments):
        start_sec = entry['start']
        end_sec = entry['end']
        speaker = entry['speaker']
        
        # Calculate sample indices
        start_sample = int(start_sec * sr)
        end_sample = int(end_sec * sr)
        
        # Slice the audio array
        chunk = y[start_sample:end_sample]
        
        # Accumulate for speaker-wise isolation
        if speaker not in speaker_audio_data:
            speaker_audio_data[speaker] = []
        speaker_audio_data[speaker].append(chunk)
        
        # If user wants individual turns exported
        if export_individual_turns:
            turns_folder = os.path.join(specific_isolated_folder, "individual_turns", speaker)
            os.makedirs(turns_folder, exist_ok=True)
            
            start_str = format_time_filename(start_sec)
            end_str = format_time_filename(end_sec)
            turn_filename = f"{audio_name_only}_{speaker}_turn_{idx+1:03d}_{start_str}_to_{end_str}.wav"
            turn_filepath = os.path.join(turns_folder, turn_filename)
            sf.write(turn_filepath, chunk, sr)
            
    # 2. Export speaker-wise concatenated audio
    if isolate_speaker_wise:
        print("\n--- Exporting Concatenated Speaker-Wise Audio ---")
        for speaker, chunks in speaker_audio_data.items():
            if chunks:
                # Concatenate all chunks for this speaker
                concatenated_audio = np.concatenate(chunks)
                speaker_filename = f"{audio_name_only}_{speaker}_isolated.wav"
                speaker_filepath = os.path.join(specific_isolated_folder, speaker_filename)
                sf.write(speaker_filepath, concatenated_audio, sr)
                print(f"[SUCCESS] Exported isolated audio for {speaker} to '{speaker_filepath}'")
                
    print(f"\n[SUCCESS] Audio splitting and speaker isolation complete! Output saved in: '{specific_isolated_folder}'")